# 03 — Adnotare Manuală Crops Parcuri

Review **interactiv** al crops-urilor auto-clasificate de B2 (pseudo-labels).

**Workflow:**
1. Fiecare crop e afișat cu clasa curentă (pseudo-label)
2. Confirmi clasa corectă cu butoanele de mai jos
3. La final: re-split → rebuild mixed_cls → retrain B3

**Clase:** `glass | metal | other | paper | plastic`

> Dacă nu ești sigur, alege **Skip** — crop-ul rămâne în clasa curentă.

In [1]:
import shutil
import json
from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as widgets

REPO_ROOT  = Path('../..').resolve()
CROPS_DIR  = REPO_ROOT / 'datasets' / 'parks_cls_unsorted' / 'all'
REVIEW_LOG = REPO_ROOT / 'datasets' / 'parks_cls_unsorted' / 'review_log.json'
CLASSES    = ['glass', 'metal', 'other', 'paper', 'plastic']
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

# Colectează toate crops-urile cu clasa curentă
all_crops = []
for cls in CLASSES:
    cls_dir = CROPS_DIR / cls
    if not cls_dir.exists():
        continue
    for p in sorted(cls_dir.iterdir()):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            all_crops.append({'path': p, 'current_cls': cls, 'assigned_cls': cls})

# Încarcă log anterior (dacă există) — rezumă de unde am rămas
reviewed = set()
if REVIEW_LOG.exists():
    with REVIEW_LOG.open(encoding='utf-8') as f:
        log_data = json.load(f)
    reviewed = set(log_data.get('reviewed', []))
    print(f'Progres anterior: {len(reviewed)}/{len(all_crops)} crop-uri review-uite')
else:
    log_data = {'reviewed': [], 'moves': []}

# Crops rămase de review
pending = [c for c in all_crops if str(c['path']) not in reviewed]

print(f'Total crops: {len(all_crops)}')
print(f'De review: {len(pending)}')
for cls in CLASSES:
    n = sum(1 for c in all_crops if c['current_cls'] == cls)
    print(f'  {cls:<10} {n:>4}')

Total crops: 284
De review: 284
  glass        12
  metal        92
  other         3
  paper       136
  plastic      41


In [2]:
from IPython.display import Image as IPImage

if not pending:
    print('[OK] Toate crop-urile au fost review-uite! Rulează celula de aplicare.')
else:
    state = {'idx': 0, 'moves': list(log_data.get('moves', []))}

    # ── Widgets ───────────────────────────────────────────────────────────────
    counter_lbl  = widgets.Label()
    current_lbl  = widgets.HTML()
    img_out      = widgets.Output(layout=widgets.Layout(width='320px', height='320px'))

    btn_skip = widgets.Button(description='Skip (OK)', button_style='success',
                              layout=widgets.Layout(width='110px'))
    class_btns = [widgets.Button(description=cls,
                                 layout=widgets.Layout(width='100px'),
                                 button_style='warning' if cls != 'other' else 'info')
                  for cls in CLASSES]
    btn_back = widgets.Button(description='← Înapoi', button_style='',
                              layout=widgets.Layout(width='100px'))
    btn_finish = widgets.Button(description='Salvează & Ieși', button_style='danger',
                                layout=widgets.Layout(width='150px'))

    top_row    = widgets.HBox([counter_lbl, current_lbl])
    action_row = widgets.HBox([btn_skip] + class_btns)
    nav_row    = widgets.HBox([btn_back, btn_finish])
    ui         = widgets.VBox([top_row, img_out, action_row, nav_row])

    def save_log():
        log_data['reviewed'] = list(reviewed | {str(pending[i]['path'])
                                                 for i in range(state['idx'])})
        log_data['moves'] = state['moves']
        with REVIEW_LOG.open('w', encoding='utf-8') as f:
            json.dump(log_data, f, indent=2)

    def show_current():
        idx = state['idx']
        if idx >= len(pending):
            with img_out:
                clear_output()
                print('[DONE] Toate crop-urile review-uite!')
            counter_lbl.value = f'{len(pending)}/{len(pending)} complete'
            return
        crop = pending[idx]
        counter_lbl.value = f'{idx+1}/{len(pending)}  '
        current_lbl.value = (f'<b>Clasa curentă: '
                             f'<span style="color:#e67e22">{crop["current_cls"]}</span></b>'
                             f'  <small style="color:#888">{crop["path"].name}</small>')
        with img_out:
            clear_output(wait=True)
            display(IPImage(filename=str(crop['path']), width=300))

    def on_skip(b):
        reviewed.add(str(pending[state['idx']]['path']))
        state['idx'] += 1
        show_current()

    def make_class_handler(target_cls):
        def handler(b):
            crop = pending[state['idx']]
            src_cls = crop['current_cls']
            if target_cls != src_cls:
                state['moves'].append({
                    'file': crop['path'].name,
                    'from': src_cls,
                    'to': target_cls
                })
            reviewed.add(str(crop['path']))
            state['idx'] += 1
            show_current()
        return handler

    def on_back(b):
        if state['idx'] > 0:
            state['idx'] -= 1
            p = str(pending[state['idx']]['path'])
            reviewed.discard(p)
            state['moves'] = [m for m in state['moves']
                               if m['file'] != pending[state['idx']]['path'].name]
            show_current()

    def on_finish(b):
        save_log()
        corrections = len(state['moves'])
        print(f'[SALVAT] {len(reviewed)} review-uite, {corrections} corecții. '
              f'Rulează celula de aplicare.')

    btn_skip.on_click(on_skip)
    btn_back.on_click(on_back)
    btn_finish.on_click(on_finish)
    for btn, cls in zip(class_btns, CLASSES):
        btn.on_click(make_class_handler(cls))

    show_current()
    display(ui)

---
## Aplicare corecții
Mută fișierele corectate în noul subfolder de clasă. **Rulează după ce ai terminat review-ul.**

In [3]:
if not REVIEW_LOG.exists():
    print('[SKIP] Niciun review log găsit. Rulează celula de review mai întâi.')
else:
    with REVIEW_LOG.open(encoding='utf-8') as f:
        log = json.load(f)

    moves = log.get('moves', [])
    print(f'Corecții de aplicat: {len(moves)}')

    applied, skipped = 0, 0
    for move in moves:
        src = CROPS_DIR / move['from'] / move['file']
        dst = CROPS_DIR / move['to']   / move['file']
        if not src.exists():
            # poate deja mutat
            if dst.exists():
                skipped += 1
                continue
            print(f'  [WARN] Lipsă: {src}')
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(src), str(dst))
        print(f'  {move["from"]} → {move["to"]}  {move["file"]}')
        applied += 1

    print(f'\nAplicat: {applied}  |  Deja OK: {skipped}')
    print('\nDistribuție după corecții:')
    for cls in CLASSES:
        n = sum(1 for p in (CROPS_DIR / cls).iterdir()
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS) if (CROPS_DIR / cls).exists() else 0
        print(f'  {cls:<10} {n:>4}')

[SKIP] Niciun review log găsit. Rulează celula de review mai întâi.


---
## Rebuild parks_cls → mixed_cls → Retrain B3
**Rulează după ce ai aplicat corecțiile.**

In [ ]:
import subprocess, sys

py = sys.executable
PARKS_CLS  = REPO_ROOT / 'datasets' / 'parks_cls'
TRASHNET   = REPO_ROOT / 'datasets' / 'trashnet_cls'
MIXED_CLS  = REPO_ROOT / 'datasets' / 'mixed_cls'
MANIFEST   = REPO_ROOT / 'datasets' / 'parks_cls_unsorted' / 'crops_manifest.csv'

def run(cmd):
    """Rulează comanda și streamează output-ul în celulă în timp real."""
    print('>', ' '.join(str(c) for c in cmd), flush=True)
    proc = subprocess.Popen(
        cmd, cwd=str(REPO_ROOT),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, encoding='utf-8', errors='replace'
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Comandă eșuată (cod {proc.returncode})')
    print()

# 1. Re-split parks_cls
run([py, 'scripts/split_classification_dataset.py',
     '--source-root', str(CROPS_DIR),
     '--out-root',    str(PARKS_CLS),
     '--manifest',    str(MANIFEST),
     '--group-by', 'source-image',
     '--val', '0.15', '--test', '0.15', '--seed', '42', '--copy', '--clear'])

# 2. Rebuild mixed_cls
run([py, 'scripts/merge_classification_datasets.py',
     '--datasets', str(TRASHNET), str(PARKS_CLS),
     '--out-dir', str(MIXED_CLS)])

# 3. Stats
run([py, 'scripts/report_classification_dataset_stats.py',
     '--data', str(MIXED_CLS)])

print('[OK] Dataset rebuilt. Rulează celula de antrenare B3.')


> d:\TrashDetectionSystem\.venv\Scripts\python.exe scripts/split_classification_dataset.py --source-root D:\TrashDetectionSystem\datasets\parks_cls_unsorted\all --out-root D:\TrashDetectionSystem\datasets\parks_cls --manifest D:\TrashDetectionSystem\datasets\parks_cls_unsorted\crops_manifest.csv --group-by source-image --val 0.15 --test 0.15 --seed 42 --copy --clear

> d:\TrashDetectionSystem\.venv\Scripts\python.exe scripts/merge_classification_datasets.py --datasets D:\TrashDetectionSystem\datasets\trashnet_cls D:\TrashDetectionSystem\datasets\parks_cls --out-dir D:\TrashDetectionSystem\datasets\mixed_cls

> d:\TrashDetectionSystem\.venv\Scripts\python.exe scripts/report_classification_dataset_stats.py --data D:\TrashDetectionSystem\datasets\mixed_cls

[OK] Dataset rebuilt. Rulează celula de antrenare B3.


In [1]:
# Antrenare B3 cu date curate
run([py, 'scripts/train_classifier.py',
     '--model', 'yolov8n-cls.pt',
     '--data',  str(MIXED_CLS),
     '--epochs', '80',
     '--imgsz', '224',
     '--batch', '32',
     '--workers', '0',
     '--device', '0',
     '--patience', '25',
     '--name', 'parks-cls-B3-clean',
     '--val'])

print('[DONE] B3 clean antrenat!')

NameError: name 'run' is not defined

In [ ]:
# Evaluare B3 clean vs B2
for name, model_path in [
    ('B2 (trashnet)',      'runs/classify/parks-cls-B2/weights/best.pt'),
    ('B3-clean (mixed)',   'runs/classify/parks-cls-B3-clean/weights/best.pt'),
]:
    print(f'\n=== {name} ===')
    run([py, 'scripts/evaluate_classifier.py',
         '--model', str(REPO_ROOT / model_path),
         '--data',  str(MIXED_CLS),
         '--split', 'test',
         '--imgsz', '224',
         '--device', '0',
         '--workers', '0',
         '--name', f'eval-{name.split()[0].lower()}-mixed-test-clean'])